# Drug Sensitivity Prediction Using SCAD

This notebook demonstrates the complete workflow for drug sensitivity prediction using SCAD (Single Cell Analysis for Drug sensitivity) with foundation models.

**Overview:**
- SCAD evaluates single-cell foundation models (scFMs) for drug sensitivity prediction
- Uses both bulk and single-cell embeddings to train the model
- Performs 5-fold cross-validation for robust evaluation
- Compares models with and without scFM-extracted embeddings


## 1. Import Required Libraries and Setup

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set working directory
os.chdir('/home/wujialu/allenwang233/scFM-Bench/DrugSensitivity')
print(f"Current working directory: {os.getcwd()}")
DRUG_NAME = "Sorafenib"
MODEL_NAME = "xTrimoGene"

Current working directory: /home/wujialu/allenwang233/scFM-Bench/DrugSensitivity


## 2. Data Preparation

### 2.1 Convert CSV to H5AD Format
First, we need to transform the data format for compatibility with our unified cell embedding extraction method.

In [3]:
# Convert CSV data to H5AD format
def convert_csv_to_h5ad():
    """
    Converts CSV files to H5AD format for compatibility with scFM embedding extraction
    """
    try:
        result = subprocess.run(['python', './utils/csv_to_h5ad.py'], 
                              capture_output=True, text=True, check=True)
        print("✓ Successfully converted CSV to H5AD format")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error converting CSV to H5AD: {e}")
        print(f"Error output: {e.stderr}")

convert_csv_to_h5ad()

✓ Successfully converted CSV to H5AD format



### 2.2 Extract Cell Embeddings Using Foundation Models
SCAD requires both bulk and single-cell embeddings. 

For Detailed Instructions, See `notebooks/2_extract_cell_embeddings.ipynb`

### 2.3 Split Data for 5-Fold Cross-Validation
We need to split the data for both scenarios: with and without embeddings.

In [ ]:
def split_data_for_cv(drug_name, model_name, with_embedding=True):
    """
    Split data for 5-fold cross-validation
    
    Args:
        drug_name: Name of the drug
        model_name: Name of the foundation model
        with_embedding: Whether to use embeddings (1) or not (0)
    """
    os.chdir('./data/split_norm/')
    
    emb_flag = 1 if with_embedding else 0
    emb_text = "with" if with_embedding else "without"
    
    cmd = [
        'python', 'split_data_SCAD_5fold_norm.py',
        '--drug', drug_name,
        '--emb', str(emb_flag),
        '--software', model_name
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"✓ Successfully split data for {drug_name} {emb_text} embeddings")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error splitting data: {e}")
        print(f"Error output: {e.stderr}")
    finally:
        os.chdir('../../')

# Split data without embeddings
print("Splitting data without embeddings...")
split_data_for_cv(DRUG_NAME, MODEL_NAME, with_embedding=False)

# Split data with embeddings
print("\nSplitting data with embeddings...")
split_data_for_cv(DRUG_NAME, MODEL_NAME, with_embedding=True)

## 3. SCAD Model Training

### 3.1 Basic SCAD Training
Train the SCAD model with specified hyperparameters.

In [ ]:
def train_scad_model(drug_name, model_name, with_embedding=True, **kwargs):
    """
    Train SCAD model with specified parameters
    
    Args:
        drug_name: Drug name for training
        model_name: Foundation model name
        with_embedding: Whether to use embeddings
        **kwargs: Additional hyperparameters
    """
    # Default hyperparameters
    params = {
        'experiment': 'FX',
        'drug': drug_name,
        'gene_set': '_norm',
        'seed': 42,
        'hidden_dim': 1024,
        'latent_dim': 128,
        'epochs': 10,
        'lambda1': 2,
        'batch_size_source': 8,
        'batch_size_target': 8,
        'embedding': 1 if with_embedding else 0,
        'software': model_name
    }
    
    # Update with any provided kwargs
    params.update(kwargs)
    
    # Build command
    cmd = ['python', 'util/SCAD_train_binarized_5folds-pub.py']
    for key, value in params.items():
        if key == 'experiment':
            cmd.extend(['-e', str(value)])
        elif key == 'drug':
            cmd.extend(['-d', str(value)])
        elif key == 'gene_set':
            cmd.extend(['-g', str(value)])
        elif key == 'seed':
            cmd.extend(['-s', str(value)])
        elif key == 'hidden_dim':
            cmd.extend(['-h_dim', str(value)])
        elif key == 'latent_dim':
            cmd.extend(['-z_dim', str(value)])
        elif key == 'epochs':
            cmd.extend(['-ep', str(value)])
        elif key == 'lambda1':
            cmd.extend(['-la1', str(value)])
        elif key == 'batch_size_source':
            cmd.extend(['-mbS', str(value)])
        elif key == 'batch_size_target':
            cmd.extend(['-mbT', str(value)])
        elif key == 'embedding':
            cmd.extend(['-emb', str(value)])
        elif key == 'software':
            cmd.extend(['--software', str(value)])
    
    emb_text = "with" if with_embedding else "without"
    print(f"Training SCAD model {emb_text} embeddings...")
    print(f"Command: {' '.join(cmd)}")
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"✓ Successfully trained SCAD model {emb_text} embeddings")
        print(result.stdout)
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Error training SCAD model: {e}")
        print(f"Error output: {e.stderr}")
        return False

# Train model without embeddings (baseline)
success_baseline = train_scad_model(DRUG_NAME, MODEL_NAME, with_embedding=False)

# Train model with embeddings
success_with_emb = train_scad_model(DRUG_NAME, MODEL_NAME, with_embedding=True)

## 4. Results Analysis and Visualization

### 4.1 Load and Compare Results
Compare the performance of models trained with and without foundation model embeddings.

In [ ]:
import re

def load_results(model_names, drug_names, gene_set="_norm"):
    """
    Load training results for comparison
    
    Args:
        model_names: Foundation model names list
        drug_names: Drug names list

    Returns:
        Dictionary containing results for both scenarios
    """
    results = {}
    for model_name in model_names:
        for drug_name in drug_names:
            embedding_results_path = f"./emb_results/{gene_set}/{drug_name}/{model_name}_train_test_summary_5folds_5seeds.txt"
            # Read the last line of the results file
            with open(embedding_results_path, "r") as f:
                lines = f.readlines()
                last_line = lines[-1].strip()
            # Example line: " BULK AUC:0.85 BULK APR:0.80 Average Test AUC = 0.83; Average Test Precision Recall = 0.78;  path = ..."
            match_auc = re.search(r'Average Test AUC = ([\d\.]+);', last_line)
            match_apr = re.search(r'Average Test Precision Recall = ([\d\.]+);', last_line)
            avgAUC = float(match_auc.group(1)) if match_auc else None
            avgAPR = float(match_apr.group(1)) if match_apr else None
            results[(model_name, drug_name)] = {"AUROC": avgAUC, "AUPR": avgAPR}
    for drug_name in drug_names:
        baseline_results_path = f"./results/{gene_set}/{drug_name}/SCAD_baseline_train_test_summary_5folds_5seeds.txt"
        with open(baseline_results_path, "r") as f:
            lines = f.readlines()
            last_line = lines[-1].strip()
        match_auc = re.search(r'Average Test AUC = ([\d\.]+);', last_line)
        match_apr = re.search(r'Average Test Precision Recall = ([\d\.]+);', last_line)
        avgAUC = float(match_auc.group(1)) if match_auc else None
        avgAPR = float(match_apr.group(1)) if match_apr else None
        results[("Raw", drug_name)] = {"AUROC": avgAUC, "AUPR": avgAPR}
    # Convert results dict to DataFrame
    df_results = []
    for (model, drug), metrics in results.items():
        df_results.append({
            "Drug": drug,
            "AUROC": metrics["AUROC"],
            "AUPR": metrics["AUPR"],
            "Model": model
        })
    result = pd.DataFrame(df_results)
    return result

# Load results
#MODEL_NAMES = ["Geneformer", "scGPT", "UCE", "scFoundation", "LangCell", "scCello"]
#DRUG_NAMES = ["Sorafenib", "Etoposide", "NVP-TAE684", "PLX4720_451Lu"]
MODEL_NAMES = ["xTrimoGene"]
DRUG_NAMES = ["Sorafenib"]
result = load_results(MODEL_NAMES, DRUG_NAMES, gene_set="_norm")

### 4.2 Create Performance Visualization
Generate plots to compare model performance with and without foundation model embeddings.

In [ ]:
model_order_scFM = ["Raw", "Geneformer", "scGPT", "UCE", "scFoundation", "LangCell", "scCello"]
colors = ["#999999", "#D6D6D6", "#E6CECF", "#A8CFC1", "#E89DA0", "#88CEE6", "#F6C8A8", "#B696B6", "#9FBA95", "#F2E1A7"]
custom_palette = {model: color for model, color in zip(model_order_scFM, colors[:len(model_order_scFM)])}
shapes = ["h", "v", "D", "H", "o", "s", "p", "*", "^", "d"]
custom_shape = {model: shape for model, shape in zip(model_order_scFM, shapes[:len(model_order_scFM)])}

In [ ]:
%matplotlib inline
# sns.stripplot(result, x="Drug", y="AUROC", hue="Model", s=10, 
#               palette=custom_palette)
result = pd.DataFrame()
sns.scatterplot(result, x="Drug", y="AUROC", style="Model", hue="Model", s=100, 
                palette=custom_palette, markers=custom_shape, alpha=0.8, edgecolor=None)
sns.despine(top=True, right=True, left=False, bottom=False)
ax = plt.gca()
# ax.legend(loc="upper left", ncol=3, title="Model", frameon=True, bbox_to_anchor=(1, 0.8))
ax.legend(loc="upper left", ncol=2, fontsize=8, bbox_to_anchor=(0.5, 1))
plt.show()

In [ ]:
%matplotlib inline
sns.set_style('white')
sns.scatterplot(result, x="Drug", y="AUPR", style="Model", hue="Model", s=100, 
                palette=custom_palette, markers=custom_shape, alpha=0.8, edgecolor=None)
sns.despine(top=True, right=True, left=False, bottom=False)
ax = plt.gca()
# ax.legend(loc="upper left", ncol=1, title="Model", frameon=True, bbox_to_anchor=(1, 0.8))
ax.legend(loc="upper left", ncol=2, fontsize=8, bbox_to_anchor=(0.5, 1))
plt.show()